### Load the SigmaProfile Prediction Model

In [51]:
import pandas as pd
import pickle as pkl
import numpy as np
import torch
from train_SP_with_CV import SigmaProfileGCN, AccumulationMeter, dataset_Sigma_Profiles, CustomLoss, collate_SP
from torch.utils.data.sampler import SubsetRandomSampler
import torch.nn as nn

In [28]:
### establish the hyperparameters used for the training/validation stage
#test_dataset = pd.read_csv("C:/Users/kverg/GDI-NN/SigmaProfileModel/Databases/MMFF_spDatabase_Test.csv")
test_dataset_path = "C:/Users/kverg/GDI-NN/SigmaProfileModel/Databases/MMFF_spDatabase_Test.csv"

model_type = "Sigma_profile_prediction"
batch_size = 16
mlp_activation = "relu"
enc_activation = "relu"
lr = 1.51e-3
use_lr_scheduler = False
epochs = 700
early_stopping = False
l2_coef = 7.79e-6
seed = 2021
data = "MMFF_sp_Database"
wandb_logs = False

save_read = f"_{data}_{model_type}_act{mlp_activation}_encAct{enc_activation}_lrsched{use_lr_scheduler}_epochs{epochs}_lr{lr}_L2coef_{l2_coef}_batchsize{batch_size}_earlystopping_{early_stopping}"

In [29]:
filename = save_read + ".pth"
dir = "C:/Users/kverg/GDI-NN/SigmaProfileModel/results_SP/"
path = dir + f"Complete_trainset_final_model_" + filename
model = SigmaProfileGCN(in_dim=50).cuda()
model.load_state_dict(torch.load(path)["model_state_dict"])

C:\Users\kverg\AppData\Local\Temp\ipykernel_30700\3528656250.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(path)["model_state_dict"])


<All keys matched successfully>

In [66]:
a = np.array([[1,2,3],[4,5,6]])
a[:,None]

array([[[1, 2, 3]],

       [[4, 5, 6]]])

In [30]:
# read dataset file
dataset = dataset_Sigma_Profiles(
    input_file_path=test_dataset_path,
    generate_all=True)
dataset_size = len(dataset)
        
# print dataset size
print('dataset size: {}'.format(dataset_size))

test_indices = np.arange(dataset_size)
# Dataloader
test_sampler = SubsetRandomSampler(test_indices)
test_loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size,
                                                sampler=test_sampler,
                                                collate_fn=collate_SP,
                                                shuffle=False,
                                                drop_last=True)
        

c:\Users\kverg\miniforge3\envs\GDINN1\lib\site-packages\dgl\heterograph.py:92: DGLWarning: Recommend creating graphs by `dgl.graph(data)` instead of `dgl.DGLGraph(data)`.
  dgl_warning(


dataset size: 200


In [31]:
def test(test_loader, model):
    stage = 'test'
    loss_fn1 = nn.MSELoss()
    loss_fn2 = nn.L1Loss()
    loss_fn3 = CustomLoss()
    loss1_accum = AccumulationMeter()
    loss2_accum = AccumulationMeter()
    loss3_accum = AccumulationMeter()
    test_pred = torch.tensor([]).cpu()
    test_true = torch.tensor([]).cpu()
    model.eval()
    with torch.set_grad_enabled(True):
        for i, component_data in enumerate(test_loader):
            sigma_profile = component_data['SP'].float().cuda() 
            output = None
            with torch.backends.cudnn.flags(enabled=False):
                output = model(component_data)
            loss1 = loss_fn1(output,sigma_profile)
            loss1_accum.update(loss1.item(),sigma_profile.size(0))
            loss2 = loss_fn2(output,sigma_profile)
            loss2_accum.update(loss2.item(),sigma_profile.size(0))
            loss3 = loss_fn3(output,sigma_profile)
            loss3_accum.update(loss3.item(),sigma_profile.size(0))
            test_pred = torch.concatenate([test_pred, output.detach().cpu()])
            test_true = torch.concatenate([test_true, sigma_profile.detach().cpu()])

    #breakpoint()
    print("[Stage {}]: RMSE={:.3f}, MAE={:.3f}, CompositeLoss={:.3f}".format(
            stage, np.sqrt(np.array(loss1_accum.avg)), loss2_accum.avg, loss3_accum.avg))

    return test_pred, test_true, [np.sqrt(np.array(loss1_accum.avg)), loss2_accum.avg, loss3_accum.avg]

In [32]:
test_pred, test_true, loss_vect = test(test_loader=test_loader,
                                       model=model)

[Stage test]: RMSE=1.596, MAE=0.582, CompositeLoss=0.851


In [56]:
def calculate_r2_scalar(y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
    """
    Calculate the scalar R2 metric for two tensors of shape [191, 52].
    
    Args:
        y_true (torch.Tensor): Ground truth tensor of shape [191, 52].
        y_pred (torch.Tensor): Predicted tensor of shape [191, 52].
        
    Returns:
        torch.Tensor: A scalar R2 metric.
    """
    # Ensure tensors are float for calculations
    y_true = y_true.float()
    y_pred = y_pred.float()
    
    # Flatten the tensors
    y_true_flat = y_true.view(-1)  # Shape: [191 * 52]
    y_pred_flat = y_pred.view(-1)  # Shape: [191 * 52]
    
    # Compute the total sum of squares
    ss_total = torch.sum((y_true_flat - torch.mean(y_true_flat))**2)
    
    # Compute the residual sum of squares
    ss_residual = torch.sum((y_true_flat - y_pred_flat)**2)
    
    # Calculate R2
    r2 = 1 - (ss_residual / ss_total)
    
    return r2

In [57]:
calculate_r2_scalar(test_pred, test_true)

tensor(0.9696)